# Kimi K3 Direct — first 500 train dialogues and full test extraction

This notebook applies the selected production configuration—**P11**, Kimi K3 through Moonshot's first-party API, and maximum reasoning effort—to the first 500 dialogues in the reformatted MathDial train dataset and every dialogue in the test dataset. Train and test are run in separate sections so either split can be stopped, resumed, or rerun independently.

One API request annotates one complete dialogue, including its synthetic `solution` unit and all real student turns. Train dialogues are ordered by numeric `dialogue_id` and limited to the first 500; no units within those dialogues are removed. Test remains complete. Valid cache records are reused; only missing, invalid, or unreadable records require another request. Cache is written to:

```text
extension/artifacts/extraction_cache/train/moonshot-direct__kimi-k3-max/P11/{dialogue_id}.json
extension/artifacts/extraction_cache/test/moonshot-direct__kimi-k3-max/P11/{dialogue_id}.json
```

These datasets do not contain human gold labels for all dialogues, so this notebook reports extraction coverage, validity, latency, and token usage—not F1 or agreement. Notebook 05 applies the usable cache records back to the misconception CSVs.


## 1. Setup

The setup locates the repository root, imports the shared data loader and production extraction backend, and checks whether `MOONSHOT_API_KEY` is available from the environment or root `.env` file. Kimi K3 fixes temperature and top-p server-side, so neither is sent.


In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'extension' / 'artifacts').exists():
        os.chdir(_candidate)
        break
else:
    raise FileNotFoundError('Run this notebook from inside the repository.')
sys.path.insert(0, str(Path.cwd()))

from extension.scripts.data_management.load_annotation_data import load_dataset
from extension.scripts.annotation import extraction, moonshot_kimi, prompt_loader

try:
    moonshot_kimi._api_key()
    KEY_FOUND = True
except RuntimeError:
    KEY_FOUND = False

print('repository:', Path.cwd())
print('MOONSHOT_API_KEY found:', KEY_FOUND)


repository: /Users/tandon.utsav2/Desktop/Experiment_1
MOONSHOT_API_KEY found: True


## 2. Load the train and test datasets and select the extraction targets

Both reformatted CSVs are loaded without converting blank cells to NaN. `dialogues_from` exposes only the final conversation transcript and ordered unit names to the model; existing annotation, correctness, KC, profile, and thread columns are not inserted into the prompt. The train records are sorted by `dialogue_id` by the shared helper and sliced to the first 500. The test target contains every unique test dialogue exactly once.


In [2]:
TRAIN_PATH = Path('data/misconception/mathdial_train.csv')
TEST_PATH = Path('data/misconception/mathdial_test.csv')

TRAIN_LIMIT = 500

train = load_dataset(TRAIN_PATH)
test = load_dataset(TEST_PATH)
ALL_TRAIN_DIALOGUES = extraction.dialogues_from(train, split='train')
TRAIN_DIALOGUES = ALL_TRAIN_DIALOGUES[:TRAIN_LIMIT]
TEST_DIALOGUES = extraction.dialogues_from(test, split='test')

assert len(ALL_TRAIN_DIALOGUES) == train['dialogue_id'].nunique()
assert len(TRAIN_DIALOGUES) == TRAIN_LIMIT
assert [dialogue['dialogue_id'] for dialogue in TRAIN_DIALOGUES] == sorted(
    train['dialogue_id'].astype(int).unique()
)[:TRAIN_LIMIT]
assert len(TEST_DIALOGUES) == test['dialogue_id'].nunique()
assert len({dialogue['dialogue_id'] for dialogue in TRAIN_DIALOGUES}) == len(TRAIN_DIALOGUES)
assert len({dialogue['dialogue_id'] for dialogue in TEST_DIALOGUES}) == len(TEST_DIALOGUES)
assert all(dialogue['split'] == 'train' for dialogue in TRAIN_DIALOGUES)
assert all(dialogue['split'] == 'test' for dialogue in TEST_DIALOGUES)

dataset_summary = pd.DataFrame({
    'source rows': {'train': len(train), 'test': len(test)},
    'source dialogues': {
        'train': len(ALL_TRAIN_DIALOGUES),
        'test': test['dialogue_id'].nunique(),
    },
    'scheduled API jobs': {
        'train': len(TRAIN_DIALOGUES),
        'test': len(TEST_DIALOGUES),
    },
    'mean units per scheduled dialogue': {
        'train': sum(len(dialogue['units']) for dialogue in TRAIN_DIALOGUES) / len(TRAIN_DIALOGUES),
        'test': sum(len(dialogue['units']) for dialogue in TEST_DIALOGUES) / len(TEST_DIALOGUES),
    },
})
display(dataset_summary.round(2))


,source rows,source dialogues,scheduled API jobs,mean units per scheduled dialogue
train,15609,2253,500,6.93
test,3902,595,595,6.56


## 3. Production configuration and cache audit helpers

`MAX_WORKERS` bounds concurrent requests within one split. The train and test sections themselves run sequentially when the notebook is executed top to bottom. A record counts as valid only when the JSON is readable and its `valid` flag is true. The audit also exposes invalid, missing, and unreadable dialogue IDs so an interrupted run can be resumed safely.


In [3]:
PROMPT = 'P11'
EFFORT = 'max'
MAX_WORKERS = 35
SLUG = moonshot_kimi.cache_slug(EFFORT)

assert PROMPT in prompt_loader.list_prompts(), f'Missing prompt file: {PROMPT}.md'

def cache_audit(dialogues):
    rows = []
    for dialogue in dialogues:
        dialogue_id = dialogue['dialogue_id']
        split = dialogue['split']
        path = extraction.cache_path(SLUG, PROMPT, dialogue_id, split)
        row = {
            'dialogue_id': dialogue_id,
            'split': split,
            'path': str(path),
            'status': 'missing',
            'latency_s': float('nan'),
            'prompt_tokens': float('nan'),
            'cached_prompt_tokens': float('nan'),
            'completion_tokens': float('nan'),
            'failure': '',
        }
        if not path.is_file():
            rows.append(row)
            continue
        try:
            record = json.loads(path.read_text())
        except (OSError, UnicodeDecodeError, json.JSONDecodeError) as exc:
            row['status'] = 'unreadable'
            row['failure'] = str(exc)
            rows.append(row)
            continue

        row['status'] = 'valid' if record.get('valid') else 'invalid'
        row['latency_s'] = record.get('latency_s', float('nan'))
        attempts = record.get('attempts') or []
        if attempts:
            attempt = attempts[-1]
            meta = attempt.get('meta') or {}
            usage = meta.get('usage') or {}
            token_summary = meta.get('tokens') or {}
            row['prompt_tokens'] = token_summary.get(
                'prompt', usage.get('prompt_tokens', float('nan'))
            )
            row['cached_prompt_tokens'] = token_summary.get(
                'cached_prompt',
                usage.get('cached_tokens',
                          (usage.get('prompt_tokens_details') or {}).get(
                              'cached_tokens', float('nan')
                          )),
            )
            row['completion_tokens'] = token_summary.get(
                'completion', usage.get('completion_tokens', float('nan'))
            )
            if row['status'] == 'invalid':
                row['failure'] = attempt.get('transport_error') or '; '.join(
                    str(error) for error in (attempt.get('errors') or [])
                )
        rows.append(row)
    return pd.DataFrame(rows).sort_values('dialogue_id').reset_index(drop=True)


def coverage_summary(audit):
    counts = audit['status'].value_counts()
    expected = len(audit)
    valid = int(counts.get('valid', 0))
    return pd.Series({
        'expected': expected,
        'valid': valid,
        'invalid': int(counts.get('invalid', 0)),
        'missing': int(counts.get('missing', 0)),
        'unreadable': int(counts.get('unreadable', 0)),
        'valid_rate': valid / expected if expected else float('nan'),
        'mean_latency_s': audit.loc[audit['status'].eq('valid'), 'latency_s'].mean(),
        'total_prompt_tokens': audit['prompt_tokens'].sum(min_count=1),
        'total_cached_prompt_tokens': audit['cached_prompt_tokens'].sum(min_count=1),
        'total_completion_tokens': audit['completion_tokens'].sum(min_count=1),
    })


def show_coverage(audit, label):
    summary = coverage_summary(audit).to_frame(label).T
    display(summary.round(3))
    issues = audit.loc[
        audit['status'].ne('valid'),
        ['dialogue_id', 'status', 'failure', 'path'],
    ]
    if not issues.empty:
        print(f'{label}: first {min(30, len(issues))} records requiring attention')
        display(issues.head(30))
    return summary

print('model:', SLUG)
print('prompt:', PROMPT)
print('reasoning effort:', EFFORT)
print('maximum concurrent workers:', MAX_WORKERS)


model: moonshot-direct/kimi-k3-max
prompt: P11
reasoning effort: max
maximum concurrent workers: 35


## 4. Extract the first 500 train dialogues

This section submits the first 500 train dialogue records, ordered by numeric `dialogue_id`, to the shared runner. Already-valid files return `cached` without an API call; invalid and missing files are requested. Keep `RUN_TRAIN_EXTRACTION=True` to run or resume this 500-dialogue extraction, or set it to `False` for a coverage-only inspection. The post-run audit is authoritative because it rereads the cache from disk.


In [ ]:
RUN_TRAIN_EXTRACTION = True

train_cache_before = cache_audit(TRAIN_DIALOGUES)
print('Train cache before extraction')
show_coverage(train_cache_before, 'train before')

train_needs_requests = train_cache_before['status'].ne('valid').any()
if RUN_TRAIN_EXTRACTION and train_needs_requests:
    if not KEY_FOUND:
        raise RuntimeError('Set MOONSHOT_API_KEY before train extraction.')
    train_run_status = moonshot_kimi.generate_annotations(
        PROMPT,
        TRAIN_DIALOGUES,
        reasoning_effort=EFFORT,
        max_workers=MAX_WORKERS,
    )
    print('Train runner statuses')
    display(pd.Series(train_run_status).value_counts().to_frame('dialogues'))
elif RUN_TRAIN_EXTRACTION:
    print('Train cache is already complete; no API requests required.')
else:
    print('Train extraction skipped: RUN_TRAIN_EXTRACTION=False')

train_cache_after = cache_audit(TRAIN_DIALOGUES)
print('Train cache after extraction')
train_coverage = show_coverage(train_cache_after, 'train after')


Train cache before extraction


,expected,valid,invalid,missing,unreadable,valid_rate,mean_latency_s,total_prompt_tokens,total_cached_prompt_tokens,total_completion_tokens
train before,500.0,30.0,0.0,470.0,0.0,0.06,1040.922,1808216.0,475392.0,997153.0


train before: first 30 records requiring attention


,dialogue_id,status,failure,path
0,0,missing,,extension/artifacts/extraction_cache/train/moo...
2,2,missing,,extension/artifacts/extraction_cache/train/moo...
3,3,missing,,extension/artifacts/extraction_cache/train/moo...
4,4,missing,,extension/artifacts/extraction_cache/train/moo...
5,5,missing,,extension/artifacts/extraction_cache/train/moo...
6,6,missing,,extension/artifacts/extraction_cache/train/moo...
7,7,missing,,extension/artifacts/extraction_cache/train/moo...
8,8,missing,,extension/artifacts/extraction_cache/train/moo...
9,9,missing,,extension/artifacts/extraction_cache/train/moo...
10,10,missing,,extension/artifacts/extraction_cache/train/moo...


  21: cached
  1: cached
  35: cached
  10: invalid
  33: invalid
  5: invalid
  36: invalid
  41: cached
  12: invalid
  28: invalid
  18: invalid
  16: invalid
  20: invalid
  8: invalid
  48: cached
  22: invalid
  38: invalid
  39: invalid
  44: invalid
  45: invalid
  43: invalid
  40: invalid
  42: invalid
  46: invalid
  47: invalid
  49: invalid
  50: invalid
  51: invalid
  53: invalid
  52: invalid
  55: invalid
  54: invalid
  56: invalid
  57: invalid
  58: invalid
  60: invalid
  62: invalid
  61: invalid
  63: invalid
  65: invalid
  64: invalid
  66: invalid
  67: invalid
  59: invalid
  69: invalid
  79: cached
  68: invalid
  71: invalid
  70: invalid
  72: invalid
  73: invalid


## 5. Extract the entire test split

This section is independent of train and writes only to the `test` cache namespace. It submits all 595 test dialogues, reuses any valid test cache, and rereads the cache after the run. Keep `RUN_TEST_EXTRACTION=True` to run or resume extraction, or set it to `False` to inspect coverage without making requests.


In [ ]:
RUN_TEST_EXTRACTION = False

test_cache_before = cache_audit(TEST_DIALOGUES)
print('Test cache before extraction')
show_coverage(test_cache_before, 'test before')

test_needs_requests = test_cache_before['status'].ne('valid').any()
if RUN_TEST_EXTRACTION and test_needs_requests:
    if not KEY_FOUND:
        raise RuntimeError('Set MOONSHOT_API_KEY before test extraction.')
    test_run_status = moonshot_kimi.generate_annotations(
        PROMPT,
        TEST_DIALOGUES,
        reasoning_effort=EFFORT,
        max_workers=MAX_WORKERS,
    )
    print('Test runner statuses')
    display(pd.Series(test_run_status).value_counts().to_frame('dialogues'))
elif RUN_TEST_EXTRACTION:
    print('Test cache is already complete; no API requests required.')
else:
    print('Test extraction skipped: RUN_TEST_EXTRACTION=False')

test_cache_after = cache_audit(TEST_DIALOGUES)
print('Test cache after extraction')
test_coverage = show_coverage(test_cache_after, 'test after')


## 6. Combined completion check

This final table compares the two independent targets: 500 selected train dialogues and all 595 test dialogues. A target is complete only when `valid == expected` and invalid, missing, and unreadable are all zero. Incomplete cache is safe: rerunning the relevant section retries only records that are not currently valid.


In [ ]:
combined_coverage = pd.concat([train_coverage, test_coverage])
combined_coverage.index = ['train', 'test']
display(combined_coverage.round(3))

complete = (
    combined_coverage['valid'].eq(combined_coverage['expected'])
    & combined_coverage[['invalid', 'missing', 'unreadable']].eq(0).all(axis=1)
)
for split, is_complete in complete.items():
    print(f'{split}:', 'COMPLETE' if is_complete else 'INCOMPLETE — rerun this section')


## Notes

- Train and test extraction run sequentially; concurrency exists only among dialogues within the active section.
- Kimi K3 uses streamed responses and the shared 300-second no-token timeout. Receiving reasoning or visible output resets that timer.
- A model response may finish transport successfully but still be invalid if it fails the P11 structural validator. Such a cache record is retained for diagnosis and retried on the next run.
- Moonshot reports token usage but not dollar cost. Token totals include all readable records currently in each cache, including records reused from earlier sessions.
- Prefix caching is controlled by Moonshot. `cached_prompt_tokens` may therefore vary by request and can be zero even when the static prompt is unchanged.
- Run Notebook 05 after extraction to validate and write the available annotations into `data/misconception/mathdial_train.csv` and `mathdial_test.csv`.
